# ============================================================
# Cell 1 — Setup
# ============================================================

In [34]:
%load_ext autoreload
%autoreload 2

import sys
import time
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

sys.path.append(str(Path.cwd().parent))
from scripts.evaluation import evaluate

RANDOM_STATE = 42
DATA_DIR    = Path.cwd().parent / 'data' / 'processed'
RESULTS_DIR = Path.cwd().parent / 'outputs' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Cell 2 — Load train/test splits
# ============================================================

In [35]:
X_train = pd.read_parquet(DATA_DIR / 'X_train.parquet')
X_test  = pd.read_parquet(DATA_DIR / 'X_test.parquet')
y_train = pd.read_parquet(DATA_DIR / 'y_train.parquet')['Victims_Condition']
y_test  = pd.read_parquet(DATA_DIR / 'y_test.parquet') ['Victims_Condition']

y_train_int, class_labels = pd.factorize(y_train, sort=True)
y_train_int = pd.Series(y_train_int, index=y_train.index)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"Class labels: {list(class_labels)}")

X_train: (370466, 21)  y_train: (370466,)
Class labels: ['With dead victims', 'With injured victims', 'Without victims']


# ============================================================
# Cell 3 — One OHE preprocessor, six models
# ============================================================

In [36]:
numeric_cols     = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['category', 'bool']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
])

def make_pipe(clf):
    return Pipeline([('pre', preprocessor), ('clf', clf)])

models = {
    'LogisticRegression': make_pipe(LogisticRegression(max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE)),
    'DecisionTree':       make_pipe(DecisionTreeClassifier(random_state=RANDOM_STATE)),
    'RandomForest':       make_pipe(RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)),
    'XGBoost':            make_pipe(XGBClassifier(tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE, verbosity=0)),
    'LightGBM':           make_pipe(LGBMClassifier(n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)),
    'CatBoost':           make_pipe(CatBoostClassifier(random_state=RANDOM_STATE, verbose=0)),
}

# ============================================================
# Cell 4 — 5-fold stratified CV on all six models
# ============================================================

In [37]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, model in models.items():
    print(f"\n {name} ")
    t0 = time.time()

    y_for_cv = y_train_int if name == 'XGBoost' else y_train
    scores = cross_val_score(model, X_train, y_for_cv, cv=cv, scoring='f1_macro', n_jobs=-1)
    elapsed = time.time() - t0

    results.append({
        'model':           name,
        'cv_macro_f1':     scores.mean(),
        'cv_macro_f1_std': scores.std(),
        'time_sec':        round(elapsed, 1),
    })
    print(f"  mean macro-F1 = {scores.mean():.4f} (± {scores.std():.4f})  |  {elapsed:.1f}s")

cv_df = pd.DataFrame(results).sort_values('cv_macro_f1', ascending=False).reset_index(drop=True)
cv_df


 LogisticRegression 
  mean macro-F1 = 0.4043 (± 0.0016)  |  20.8s

 DecisionTree 
  mean macro-F1 = 0.4353 (± 0.0016)  |  205.7s

 RandomForest 
  mean macro-F1 = 0.4063 (± 0.0017)  |  1185.0s

 XGBoost 
  mean macro-F1 = 0.4159 (± 0.0029)  |  13.5s

 LightGBM 
  mean macro-F1 = 0.3985 (± 0.0020)  |  15.1s

 CatBoost 
  mean macro-F1 = 0.4188 (± 0.0029)  |  101.7s


,model,cv_macro_f1,cv_macro_f1_std,time_sec
0,DecisionTree,0.435274,0.001588,205.7
1,CatBoost,0.418813,0.002869,101.7
2,XGBoost,0.415918,0.002852,13.5
3,RandomForest,0.406280,0.001665,1185.0
4,LogisticRegression,0.404345,0.001648,20.8
5,LightGBM,0.398456,0.002037,15.1


# ============================================================
# Cell 5 — Save results + pick top 3 for Phase 3
# ============================================================

In [38]:
cv_df.to_csv(RESULTS_DIR / 'phase2_six_classifiers.csv', index=False)

top_3 = cv_df.head(3)['model'].tolist()
print(f"\nTop 3 classifiers by CV macro-F1: {top_3}")

with open(RESULTS_DIR / 'phase2_top3.txt', 'w') as f:
    f.write('\n'.join(top_3))


Top 3 classifiers by CV macro-F1: ['DecisionTree', 'CatBoost', 'XGBoost']
